In [ ]:
import re
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import nltk

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    pipeline,
)
from datasets import Dataset

# ── NLTK Downloads ──────────────────────────────────────────────────────
nltk.download("punkt",       quiet=True)
nltk.download("punkt_tab",   quiet=True)
nltk.download("stopwords",   quiet=True)

# ── Config ──────────────────────────────────────────────────────
# Using distilroberta-base is 2x faster than roberta-base and keeps high accuracy
MODEL_NAME   = "distilroberta-base"
MAX_LEN      = 128        
BATCH_SIZE   = 32         
EPOCHS       = 3          
LR           = 3e-5
NUM_LABELS   = 2
OUTPUT_DIR   = "./roberta-mental-health"
SAVE_PATH    = "./roberta-mental-health-final"
RANDOM_SEED  = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ╔═══════════════════════════════════════════════════════════
# 1.  LOAD DATASETS
# ════════════════════════════════════════════════════════════
def load_train_data(path1: str = "/content/Dataset 1.csv", 
                    path2: str = "/content/dataset 2.csv") -> pd.DataFrame:
    df1 = pd.read_csv(path1, engine="python")
    df2 = pd.read_csv(path2, engine="python")

    # Standardise column names
    df1 = df1[["statement", "status"]].rename(columns={"statement": "text", "status": "label"})
    df2 = df2[["post_text", "label"]].rename(columns={"post_text": "text"})

    # Map df1 labels → binary
    df1["label"] = df1["label"].str.lower().map({
        "depression":           1,
        "suicidal":             1,
        "anxiety":              1,
        "stress":               1,
        "bipolar":              1,
        "personality disorder": 1,
        "normal":               0,
    })
    df2["label"] = df2["label"].astype(int)

    df = pd.concat([df1, df2], ignore_index=True)
    df.dropna(subset=["text", "label"], inplace=True)
    df.drop_duplicates(subset="text", inplace=True)
    df["label"] = df["label"].astype(int)

    print(f"Train Dataset shape : {df.shape}")
    print(f"Train Label counts  :\n{df['label'].value_counts()}\n")
    return df

def load_test_data(path3: str = "/content/ds 3.csv") -> pd.DataFrame:
    df = pd.read_csv(path3, engine="python")
    # Dataset 3 has 'sentence' and 'label' columns
    df = df[["sentence", "label"]].rename(columns={"sentence": "text"})
    df.dropna(subset=["text", "label"], inplace=True)
    df.drop_duplicates(subset="text", inplace=True)
    df["label"] = df["label"].astype(int)
    
    print(f"Test Dataset shape : {df.shape}")
    print(f"Test Label counts  :\n{df['label'].value_counts()}\n")
    return df

# ╔═══════════════════════════════════════════════════════════
# 2.  TEXT CLEANING
# ════════════════════════════════════════════════════════════
def clean_text_bert(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+",        "",  text)   # remove URLs
    text = re.sub(r"@\w+",           "",  text)   # remove @mentions
    text = re.sub(r"#\w+",           "",  text)   # remove #hashtags
    text = re.sub(r"[^a-zA-Z .!?,]", " ", text)  # keep punctuation for BERT
    text = re.sub(r"\s+",            " ", text).strip()
    return text


# ╔═══════════════════════════════════════════════════════════
# 3.  TOKENISE
# ════════════════════════════════════════════════════════════
def build_hf_datasets(
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test,
    tokenizer,
):
    def tokenize_batch(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
        )

    splits = {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
    }
    datasets = {}
    for name, (X, y) in splits.items():
        ds = Dataset.from_dict({"text": X, "labels": y})
        ds = ds.map(tokenize_batch, batched=True, remove_columns=["text"])
        ds.set_format("torch")
        datasets[name] = ds

    return datasets["train"], datasets["val"], datasets["test"]


# ╔═══════════════════════════════════════════════════════════
# 4.  CLASS-WEIGHTED TRAINER 
# ════════════════════════════════════════════════════════════
class WeightedTrainer(Trainer):
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss    = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


# ╔═══════════════════════════════════════════════════════════
# 5.  METRICS
# ════════════════════════════════════════════════════════════
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds, average="weighted"),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


# ╔═══════════════════════════════════════════════════════════
# 6.  MAIN
# ════════════════════════════════════════════════════════════
def main():
    # ── Load Train & Val data ────────────────────────────────
    df_train_full = load_train_data()
    df_train_full["text"] = df_train_full["text"].apply(clean_text_bert)
    df_train_full = df_train_full[df_train_full["text"].str.strip() != ""]

    X_train_full = df_train_full["text"].tolist()
    y_train_full = df_train_full["label"].tolist()

    # Split 10% of training data for validation to allow Early Stopping
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.1, random_state=RANDOM_SEED, stratify=y_train_full
    )
    
    # ── Load Test data (Dataset 3) ───────────────────────────
    df_test = load_test_data()
    df_test["text"] = df_test["text"].apply(clean_text_bert)
    df_test = df_test[df_test["text"].str.strip() != ""]
    
    X_test = df_test["text"].tolist()
    y_test = df_test["label"].tolist()

    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}\n")

    # ── Tokeniser ─────────────────────────────────────────
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds, val_ds, test_ds = build_hf_datasets(
        X_train, y_train, X_val, y_val, X_test, y_test, tokenizer
    )

    # ── Model ───────────────────────────────────────
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={0: "Normal", 1: "MentalHealth"},
        label2id={"Normal": 0, "MentalHealth": 1},
    ).to(device)

    # ── Class weights ───────────────────────────
    label_counts  = Counter(y_train)
    total         = sum(label_counts.values())
    class_weights = torch.tensor(
        [total / (NUM_LABELS * label_counts[i]) for i in range(NUM_LABELS)],
        dtype=torch.float,
    ).to(device)
    print(f"Class weights: {class_weights}\n")

    # ── Training arguments ────────────────────────
    training_args = TrainingArguments(
        output_dir                  = OUTPUT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE * 2,
        learning_rate               = LR,
        weight_decay                = 0.01,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        greater_is_better           = True,
        fp16                        = torch.cuda.is_available(),
        logging_steps               = 100,
        report_to                   = "none",
        seed                        = RANDOM_SEED,
    )

    # ── Train ────────────────────────────────────────────
    trainer = WeightedTrainer(
        class_weights   = class_weights,
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
    )

    print("=" * 60)
    print("Starting fine-tuning …")
    print("=" * 60)
    trainer.train()

    # ── Evaluate ─────────────────────────────────
    print("\n=== Validation Results ===")
    val_results = trainer.evaluate(val_ds)
    for k, v in val_results.items():
        print(f"  {k}: {v:.4f}")

    print("\n=== Test Set Results (Dataset 3) ===")
    test_results = trainer.evaluate(test_ds)
    for k, v in test_results.items():
        print(f"  {k}: {v:.4f}")

    preds_output = trainer.predict(test_ds)
    logits = preds_output.predictions
    y_pred = np.argmax(logits, axis=-1)
    print("\n=== Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=["Normal", "MentalHealth"]))

    # ── ROC Curve ──────────────────────────────────────────────
    print("\n=== Generating ROC Diagram ===")
    # Calculate probabilities using softmax
    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    
    fpr, tpr, thresholds = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()

    # ── Save ─────────────────────────────────────────
    trainer.save_model(SAVE_PATH)
    tokenizer.save_pretrained(SAVE_PATH)
    print(f"\nModel saved to: {SAVE_PATH}")


if __name__ == "__main__":
    main()

